<a href="https://colab.research.google.com/github/camistrika/BETO_HUMOR/blob/main/notebooks/cross_validation_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validación cruzada (k-fold) — BETO + LoRA
Fija el learning_rate en el valor más estable visto en el grid search,
y explora combinaciones de `r` y `weight_decay` para elegir la mejor regularización.

In [1]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

Cloning into 'BETO_HUMOR'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 154 (delta 68), reused 154 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 3.11 MiB | 7.25 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/BETO_HUMOR
  Preparing metadata (setup.py) ... done


In [1]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.4 MB/s eta 0:00:00


In [2]:
%cd /content/BETO_HUMOR

/content/BETO_HUMOR


In [3]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer
from google.colab import files

from betohumor.config import DataConfig, BetoConfig, LoraConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.lora import build_beto_lora
from betohumor.hyperparam_search import run_one


## 1. Datos y configuraciones

In [5]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)

# Unimos train+val para repartir en folds.
df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)
label_col = data_config.label_col

Train: 19197 | Val: 2397 | Test: 2400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

## 2. Configuración de la búsqueda


In [6]:
LR_VALUES     = [1e-4, 3e-4, 1e-3]
R_VALUES      = [8, 16, 32]
WEIGHT_DECAYS = [0.01, 0.05]
N_FOLDS       = 3

params_grid = [
    {'r': r, 'lora_alpha': r * 2, 'learning_rate': lr, 'weight_decay': wd}
    for lr, r, wd in product(LR_VALUES, R_VALUES, WEIGHT_DECAYS)
]


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
print(f'Total combinaciones: {len(params_grid)}')
params_grid

Total combinaciones: 18


[{'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0003, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0003, 'weight_decay': 0.05},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0003, 'weight_decay': 0.01},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0003, 'weight_decay': 0.05},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0003, 'weight_decay': 0.01},
 {'r': 32, 'lora_alpha': 64, 'learning_rate': 0.0003, 'weight_decay': 0.05},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.001, 'weight_decay': 0.01},
 {'r'

## 3. Correr la validación cruzada

In [7]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"r{params['r']}_lr{params['learning_rate']}_wd{params['weight_decay']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_lora/{run_name}"
        macro_f1, trainer = run_one(
            lambda p: build_beto_lora(beto_config, LoraConfig(r=p['r'], lora_alpha=p['lora_alpha'])),
            params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")




--- r8_lr0.0001_wd0.01_fold1 ---


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

trainable params: 296,450 || all params: 110,148,868 || trainable%: 0.2691


...

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

trainable params: 1,181,186 || all params: 111,033,604 || trainable%: 1.0638
{'loss': '0.6422', 'grad_norm': '1.065', 'learning_rate': '6.608e-05', 'epoch': '1'}
{'eval_loss': '0.4273', 'eval_accuracy': '0.7838', 'eval_macro_f1': '0.7796', 'eval_runtime': '4.805', 'eval_samples_per_second': '1498', 'eval_steps_per_second': '11.86', 'epoch': '1'}
  Época 1 | val_loss=0.4273 | macro_f1=0.7796 | accuracy=0.7838
{'loss': '0.4019', 'grad_norm': '2.321', 'learning_rate': '0.0001327', 'epoch': '2'}
{'eval_loss': '0.3737', 'eval_accuracy': '0.8131', 'eval_macro_f1': '0.8104', 'eval_runtime': '4.816', 'eval_samples_per_second': '1495', 'eval_steps_per_second': '11.84', 'epoch': '2'}
  Época 2 | val_loss=0.3737 | macro_f1=0.8104 | accuracy=0.8131
{'loss': '0.3567', 'grad_norm': '0.8616', 'learning_rate': '0.0001994', 'epoch': '3'}
{'eval_loss': '0.3832', 'eval_accuracy': '0.8005', 'eval_macro_f1': '0.7993', 'eval_runtime': '4.799', 'eval_samples_per_second': '1500', 'eval_steps_per_second': '11.

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

trainable params: 1,181,186 || all params: 111,033,604 || trainable%: 1.0638
{'loss': '0.6247', 'grad_norm': '1.257', 'learning_rate': '6.608e-05', 'epoch': '1'}
{'eval_loss': '0.4556', 'eval_accuracy': '0.7617', 'eval_macro_f1': '0.7579', 'eval_runtime': '4.838', 'eval_samples_per_second': '1488', 'eval_steps_per_second': '11.78', 'epoch': '1'}
  Época 1 | val_loss=0.4556 | macro_f1=0.7579 | accuracy=0.7617
{'loss': '0.3979', 'grad_norm': '0.888', 'learning_rate': '0.0001327', 'epoch': '2'}
{'eval_loss': '0.3777', 'eval_accuracy': '0.8131', 'eval_macro_f1': '0.809', 'eval_runtime': '4.819', 'eval_samples_per_second': '1494', 'eval_steps_per_second': '11.83', 'epoch': '2'}
  Época 2 | val_loss=0.3777 | macro_f1=0.8090 | accuracy=0.8131
{'loss': '0.3555', 'grad_norm': '1.374', 'learning_rate': '0.0001994', 'epoch': '3'}
{'eval_loss': '0.3799', 'eval_accuracy': '0.8018', 'eval_macro_f1': '0.8', 'eval_runtime': '4.778', 'eval_samples_per_second': '1507', 'eval_steps_per_second': '11.93', 

In [8]:
print(f"\n=== r={params['r']} lr={params['learning_rate']} wd={params['weight_decay']} -> Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")


=== r=32 lr=0.001 wd=0.05 -> Media: 0.8505 ± 0.0025 ===


## 4. Resumen y guardado

In [9]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_lora.csv', index=False)

df_summary = (
    df_folds
    .groupby(['r', 'lora_alpha', 'learning_rate', 'weight_decay'])['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_lora_summary.csv', index=False)
df_summary

,r,lora_alpha,learning_rate,weight_decay,mean_macro_f1,std_macro_f1
15,32,64,0.0003,0.05,0.850698,0.006104
17,32,64,0.0010,0.05,0.850487,0.003123
6,16,32,0.0001,0.01,0.850293,0.008004
0,8,16,0.0001,0.01,0.850289,0.004723
13,32,64,0.0001,0.05,0.849713,0.000444
7,16,32,0.0001,0.05,0.849616,0.007783
10,16,32,0.0010,0.01,0.849565,0.002334
4,8,16,0.0010,0.01,0.849461,0.004124
14,32,64,0.0003,0.01,0.849376,0.003050
11,16,32,0.0010,0.05,0.849205,0.002832


In [10]:
files.download('results/cv_results_lora.csv')
files.download('results/cv_results_lora_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## 2B. Configuración de la búsqueda - weight_decay = 0.05



In [ ]:
LR_VALUES     = [1e-4, 3e-4, 1e-3]
R_VALUES      = [8, 16, 32]
WEIGHT_DECAYS = [0.05]
N_FOLDS       = 3

params_grid = [
    {'r': r, 'lora_alpha': r * 2, 'learning_rate': lr, 'weight_decay': wd}
    for lr, r, wd in product(LR_VALUES, R_VALUES, WEIGHT_DECAYS)
]

def builder(params):
    lora_config = LoraConfig(r=params['r'], lora_alpha=params['lora_alpha'])
    return build_beto_lora(beto_config, lora_config)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
params_grid

## 5. Correr la validación cruzada (weight_decay = 0.05)

In [ ]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"r{params['r']}_lr{params['learning_rate']}_wd{params['weight_decay']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_lora/{run_name}"
        macro_f1, trainer = run_one(
            builder, params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")

    print(f"\n=== r={params['r']} lr={params['learning_rate']} wd={params['weight_decay']} -> Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")

## 6. Guardado bloque B (weight_decay = 0.05)

In [ ]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_lora_wd05.csv', index=False)
df_folds

In [ ]:
files.download('results/cv_results_lora_wd05.csv')


## 7. Unir ambos bloques


In [ ]:
df_folds_wd01 = pd.read_csv('results/cv_results_lora_wd01.csv')
df_folds_wd05 = pd.read_csv('results/cv_results_lora_wd05.csv')

df_folds = pd.concat([df_folds_wd01, df_folds_wd05]).reset_index(drop=True)
df_folds.to_csv('results/cv_results_lora.csv', index=False)

df_summary = (
    df_folds
    .groupby(['r', 'lora_alpha', 'learning_rate', 'weight_decay'])['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_lora_summary.csv', index=False)
df_summary

In [ ]:
files.download('results/cv_results_lora.csv')
files.download('results/cv_results_lora_summary.csv')